# Predicting Smartphone Addiction — Advanced Pipeline (XGBoost + LightGBM + CatBoost)

This notebook is a **leaner, stronger** version of the first one. It keeps only the models
that actually push ROC-AUC higher on tabular data like this:

- **XGBoost**
- **LightGBM**
- **CatBoost** (added — usually the strongest of the three on this kind of data, and handles
  categorical columns natively)

And it upgrades the pipeline itself with the things that matter most once you're already past 0.95 AUC:

1. Careful cleaning (same as before) + **correct ordinal encoding** (a mis-encoded ordinal
   column is one of the most common hidden causes of a stuck score)
2. Feature engineering, including interaction features
3. **5-Fold Stratified Cross-Validation** instead of one train/val split (more reliable score,
   and gives you 5 models per algorithm to average — a free ensemble)
4. **Optuna hyperparameter search** for each model (a proper search beats hand-picked parameters)
5. **Out-of-fold (OOF) evaluation** — the honest way to measure ROC-AUC before submitting
6. **Weighted blend** of the 3 models' final test predictions
7. One submission file per model **and** one for the blend

> Comments explain every step. Run top to bottom.


## 0. Environment setup

```bash
conda activate addiction
pip install optuna catboost
```
(you should already have pandas, numpy, scikit-learn, xgboost, lightgbm from before)


In [1]:
# --- Core ---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# --- Preprocessing ---
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

# --- Models ---
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# --- Tuning ---
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # keeps output clean

# --- Evaluation ---
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
N_FOLDS = 5          # <- can increase to 10 for an even more robust (but slower) estimate
N_TRIALS = 30         # <- Optuna trials per model. Increase (e.g. 60-100) if you have time/compute


## 1. Load data

Update the paths if needed.


In [2]:
TRAIN_PATH = "./train.csv"
TEST_PATH  = "./test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("train:", train_df.shape)   # (691369, 13)
print("test :", test_df.shape)    # (296302, 12)


train: (691369, 14)
test : (296302, 13)


## 2. Cleaning

Same rules as before: impossible values -> NaN -> imputed with train-only statistics,
outliers clipped (not dropped, so no test row ever loses a prediction).

**New / important fix:** ordinal-looking columns (e.g. `stress_level`, `academic_work_impact`
if they're text like Low/Medium/High) are mapped to **ordered** integers instead of an
arbitrary alphabetical `LabelEncoder` order. This one bug alone can quietly cap your AUC.


In [3]:
FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
NUMERIC_COLS = [c for c in FEATURE_COLS if train_df[c].dtype != "object"]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if train_df[c].dtype == "object"]
print("Numeric:", NUMERIC_COLS)
print("Categorical:", CATEGORICAL_COLS)

# Look at the unique values of each categorical column BEFORE encoding anything,
# so we know which ones are truly ordinal (have a natural order) vs purely nominal (e.g. gender).
for col in CATEGORICAL_COLS:
    print(col, "->", train_df[col].unique())


Numeric: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']
Categorical: []


In [5]:
# ---- 4.2 Fix impossible / noisy values -> convert to NaN so they get imputed properly
# Adjust these plausible ranges if your dataset's real-world limits are different.
def clean_impossible_values(df):
    df = df.copy()

    # Convert every column used in numeric range checks, including numeric-looking text,
    # so comparisons with numeric limits are always valid.
    numeric_range_cols = [
        "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
        "work_study_hours", "sleep_hours", "weekend_screen_time",
        "notifications_per_day", "app_opens_per_day", "stress_level"
    ]
    for col in numeric_range_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "age" in df.columns:
        df.loc[(df["age"] < 5) | (df["age"] > 100), "age"] = np.nan
    for col in ["daily_screen_time_hours", "social_media_hours", "gaming_hours",
                "work_study_hours", "sleep_hours", "weekend_screen_time"]:
        if col in df.columns:
            df.loc[(df[col] < 0) | (df[col] > 24), col] = np.nan
    for col in ["notifications_per_day", "app_opens_per_day"]:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan
    if "stress_level" in df.columns:
        # assuming stress_level is on a 1-10 scale; adjust if yours differs
        df.loc[(df["stress_level"] < 0) | (df["stress_level"] > 10), "stress_level"] = np.nan
    return df

train_df = clean_impossible_values(train_df)
test_df  = clean_impossible_values(test_df)

# Refresh these lists because coercion may have changed string columns into numeric columns.
FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
NUMERIC_COLS = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(train_df[c])]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if c not in NUMERIC_COLS]

In [7]:
# ---- 4.3 Impute missing values
# Fit the imputer on TRAIN ONLY, then apply to both train and test (avoids leakage).
# keep_empty_features=True preserves columns that are entirely missing in train.
num_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
train_df[NUMERIC_COLS] = num_imputer.fit_transform(train_df[NUMERIC_COLS])
test_df[NUMERIC_COLS]  = num_imputer.transform(test_df[NUMERIC_COLS])

if CATEGORICAL_COLS:
    cat_imputer = SimpleImputer(strategy="most_frequent", keep_empty_features=True)
    train_df[CATEGORICAL_COLS] = cat_imputer.fit_transform(train_df[CATEGORICAL_COLS])
    test_df[CATEGORICAL_COLS]  = cat_imputer.transform(test_df[CATEGORICAL_COLS])

# Sanity check: no missing values left in the feature columns
print("Missing left in train features:", train_df[FEATURE_COLS].isna().sum().sum())
print("Missing left in test features :", test_df[FEATURE_COLS].isna().sum().sum())

Missing left in train features: 0
Missing left in test features : 0


In [13]:
# ---- Encode categorical columns CORRECTLY
# 1) Ordinal columns -> map to an EXPLICIT, meaningful order (edit these dicts to match
#    the exact category strings printed above in the "unique values" cell)
ORDINAL_MAPS = {
    "stress_level": {"Low": 0, "Medium": 1, "High": 2},
    "academic_work_impact": {"None": 0, "Low": 1, "Medium": 2, "High": 3},
}

for col, mapping in ORDINAL_MAPS.items():
    if col in train_df.columns and train_df[col].dtype == "object":
        train_df[col] = train_df[col].map(mapping)
        test_df[col]  = test_df[col].map(mapping)
        # any leftover NaN (unseen category) -> fill with the median ordinal value
        med = train_df[col].median()
        train_df[col] = train_df[col].fillna(med)
        test_df[col]  = test_df[col].fillna(med)

# 2) Purely nominal columns (no natural order, e.g. gender) -> plain LabelEncoder is fine
NOMINAL_COLS = [c for c in CATEGORICAL_COLS if c not in ORDINAL_MAPS]
encoders = {}
for col in NOMINAL_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    known = set(le.classes_)
    test_df[col] = test_df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

# Guarantee that every feature passed to XGBoost/LightGBM/CatBoost is numeric.
# Unexpected text values become NaN and are filled using train-only medians.
for col in FEATURE_COLS:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

print("Ordinal columns fixed:", list(ORDINAL_MAPS.keys()))
print("Nominal columns encoded:", NOMINAL_COLS)
print("Non-numeric feature columns:", train_df[FEATURE_COLS].select_dtypes(exclude=np.number).columns.tolist())

Ordinal columns fixed: ['stress_level', 'academic_work_impact']
Nominal columns encoded: ['gender']
Non-numeric feature columns: []


In [9]:
# ---- Clip outliers at 1st/99th percentile (learned on train only)
for col in NUMERIC_COLS:
    low, high = train_df[col].quantile(0.01), train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(low, high)
    test_df[col]  = test_df[col].clip(low, high)


## 3. Feature engineering

Ratio and interaction features that usually help tree-boosting models find patterns faster.


In [14]:
def add_features(df):
    df = df.copy()
    df["screen_time_ratio"]      = df["daily_screen_time_hours"] / 24
    df["social_media_ratio"]     = df["social_media_hours"] / (df["daily_screen_time_hours"] + 1e-3)
    df["gaming_ratio"]           = df["gaming_hours"] / (df["daily_screen_time_hours"] + 1e-3)
    df["notif_per_screen_hour"]  = df["notifications_per_day"] / (df["daily_screen_time_hours"] + 1e-3)
    df["opens_per_notif"]        = df["app_opens_per_day"] / (df["notifications_per_day"] + 1e-3)
    df["weekend_vs_weekday"]     = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["work_sleep_ratio"]       = df["work_study_hours"] / (df["sleep_hours"] + 1e-3)
    df["sleep_deficit"]          = 8 - df["sleep_hours"]                  # below "healthy" 8h
    if "stress_level" in df.columns:
        df["stress_x_screen"]    = df["stress_level"] * df["daily_screen_time_hours"]
    return df

train_df = add_features(train_df)
test_df  = add_features(test_df)

FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
print("Total features:", len(FEATURE_COLS))

X = train_df[FEATURE_COLS]
y = train_df["addicted_label"]
X_test = test_df[FEATURE_COLS]


Total features: 21


## 4. Optuna hyperparameter search (per model)

Each search uses a **single 80/20 split** (fast) just to find good parameters.
Once we have the best parameters, Section 5 retrains properly with **5-fold CV** for the
final, more robust model. This two-step process (quick search -> robust final training)
is much faster than running Optuna inside every fold.


In [15]:
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)


### 4.1 Tune XGBoost

In [16]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model = XGBClassifier(**params, eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_tr, y_tr)
    preds = model.predict_proba(X_va)[:, 1]
    return roc_auc_score(y_va, preds)

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best XGBoost AUC (quick search):", xgb_study.best_value)
print("Best XGBoost params:", xgb_study.best_params)


  0%|          | 0/30 [00:00<?, ?it/s]

Best XGBoost AUC (quick search): 0.9628101586752056
Best XGBoost params: {'n_estimators': 1293, 'learning_rate': 0.05956495847000891, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.9931275336917837, 'colsample_bytree': 0.8904457836394609, 'reg_alpha': 1.693336553948598, 'reg_lambda': 0.004474578269836449}


### 4.2 Tune LightGBM

In [17]:
def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model = LGBMClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_tr, y_tr)
    preds = model.predict_proba(X_va)[:, 1]
    return roc_auc_score(y_va, preds)

lgbm_study = optuna.create_study(direction="maximize")
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best LightGBM AUC (quick search):", lgbm_study.best_value)
print("Best LightGBM params:", lgbm_study.best_params)


  0%|          | 0/30 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038526 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3965
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024698 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3965
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
[LightGBM]

### 4.3 Tune CatBoost

CatBoost is often the strongest of the three on datasets like this (mixed numeric +
categorical features, moderate size). It's included here as the "better model" you asked for.


In [18]:
def catboost_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }
    model = CatBoostClassifier(**params, eval_metric="AUC", random_state=RANDOM_STATE, verbose=False)
    model.fit(X_tr, y_tr)
    preds = model.predict_proba(X_va)[:, 1]
    return roc_auc_score(y_va, preds)

cat_study = optuna.create_study(direction="maximize")
cat_study.optimize(catboost_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best CatBoost AUC (quick search):", cat_study.best_value)
print("Best CatBoost params:", cat_study.best_params)


  0%|          | 0/30 [00:00<?, ?it/s]

Best CatBoost AUC (quick search): 0.9616495980680146
Best CatBoost params: {'iterations': 1357, 'learning_rate': 0.09680771182018107, 'depth': 8, 'l2_leaf_reg': 1.1909923149016413, 'bagging_temperature': 0.13369247892758027}


## 5. Final training with 5-Fold Stratified Cross-Validation

Now we retrain each model **properly**, using its best Optuna parameters, across 5 folds.
For each fold: train on 4/5 of the data, predict on the held-out 1/5 (out-of-fold, "OOF"
predictions), and predict on the real test set (averaged over the 5 folds).

Why this matters:
- The **OOF ROC-AUC** (computed over all 691k rows, each predicted by a model that never saw it
  during training) is the most honest estimate of your real leaderboard score.
- Averaging 5 models' test predictions acts as a free ensemble, usually adding a bit of AUC
  on top of a single train/val split.


In [19]:
def run_cv(model_class, params, name, X, y, X_test, n_folds=N_FOLDS):
    """Trains model_class(**params) across n_folds, returns OOF AUC and averaged test predictions."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr_f, X_va_f = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_f, y_va_f = y.iloc[tr_idx], y.iloc[va_idx]

        model = model_class(**params)
        if name == "catboost":
            model.fit(X_tr_f, y_tr_f, verbose=False)
        else:
            model.fit(X_tr_f, y_tr_f)

        fold_pred = model.predict_proba(X_va_f)[:, 1]
        oof_preds[va_idx] = fold_pred
        fold_auc = roc_auc_score(y_va_f, fold_pred)
        fold_scores.append(fold_auc)
        print(f"  {name} fold {fold}/{n_folds} AUC = {fold_auc:.5f}")

        test_preds += model.predict_proba(X_test)[:, 1] / n_folds

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"{name} -> mean fold AUC = {np.mean(fold_scores):.5f} | OOF AUC = {oof_auc:.5f}")
    return oof_auc, oof_preds, test_preds


 xgboost fold 1/5 AUC = 0.96250
  xgboost fold 2/5 AUC = 0.96344
  xgboost fold 3/5 AUC = 0.96356
  xgboost fold 4/5 AUC = 0.96425
  xgboost fold 5/5 AUC = 0.96306
xgboost -> mean fold AUC = 0.96336 | OOF AUC = 0.96336

In [20]:
# ---- XGBoost final CV run
xgb_params = dict(xgb_study.best_params, eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1)
xgb_oof_auc, xgb_oof, xgb_test_preds = run_cv(XGBClassifier, xgb_params, "xgboost", X, y, X_test)


  xgboost fold 1/5 AUC = 0.96250
  xgboost fold 2/5 AUC = 0.96344
  xgboost fold 3/5 AUC = 0.96356
  xgboost fold 4/5 AUC = 0.96425
  xgboost fold 5/5 AUC = 0.96306
xgboost -> mean fold AUC = 0.96336 | OOF AUC = 0.96336


OOF AUC = 0.96309

In [21]:
# ---- LightGBM final CV run
lgbm_params = dict(lgbm_study.best_params, random_state=RANDOM_STATE, n_jobs=-1)
lgbm_oof_auc, lgbm_oof, lgbm_test_preds = run_cv(LGBMClassifier, lgbm_params, "lightgbm", X, y, X_test)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017200 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3970
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
  lightgbm fold 1/5 AUC = 0.96231
[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3967
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start train

  catboost fold 1/5 AUC = 0.96163
  catboost fold 2/5 AUC = 0.96216
  catboost fold 3/5 AUC = 0.96248
  catboost fold 4/5 AUC = 0.96324
  catboost fold 5/5 AUC = 0.96191
catboost -> mean fold AUC = 0.96229 | OOF AUC = 0.96228

In [22]:
# ---- CatBoost final CV run
cat_params = dict(cat_study.best_params, eval_metric="AUC", random_state=RANDOM_STATE)
cat_oof_auc, cat_oof, cat_test_preds = run_cv(CatBoostClassifier, cat_params, "catboost", X, y, X_test)


  catboost fold 1/5 AUC = 0.96163
  catboost fold 2/5 AUC = 0.96216
  catboost fold 3/5 AUC = 0.96248
  catboost fold 4/5 AUC = 0.96324
  catboost fold 5/5 AUC = 0.96191
catboost -> mean fold AUC = 0.96229 | OOF AUC = 0.96228


## 6. Compare models (OOF AUC — the trustworthy number)


      model  oof_roc_auc
0   xgboost     0.963361
1  lightgbm     0.963093
2  catboost     0.962284

In [23]:
results_df = pd.DataFrame({
    "model": ["xgboost", "lightgbm", "catboost"],
    "oof_roc_auc": [xgb_oof_auc, lgbm_oof_auc, cat_oof_auc]
}).sort_values("oof_roc_auc", ascending=False).reset_index(drop=True)

print(results_df)


      model  oof_roc_auc
0   xgboost     0.963361
1  lightgbm     0.963093
2  catboost     0.962284


## 7. Weighted blend of the 3 models

Weight each model's test predictions by its own OOF AUC, so the strongest model has the
biggest say. This usually beats every individual model and a plain average.


In [24]:
weights = {
    "xgboost": xgb_oof_auc,
    "lightgbm": lgbm_oof_auc,
    "catboost": cat_oof_auc,
}
total_weight = sum(weights.values())

blend_test_preds = (
    xgb_test_preds  * (weights["xgboost"]  / total_weight) +
    lgbm_test_preds * (weights["lightgbm"] / total_weight) +
    cat_test_preds  * (weights["catboost"] / total_weight)
)

# Estimate the blend's OOF AUC too, using the same weights on OOF predictions
blend_oof = (
    xgb_oof  * (weights["xgboost"]  / total_weight) +
    lgbm_oof * (weights["lightgbm"] / total_weight) +
    cat_oof  * (weights["catboost"] / total_weight)
)
blend_oof_auc = roc_auc_score(y, blend_oof)
print(f"Weighted blend OOF ROC-AUC = {blend_oof_auc:.5f}")


Weighted blend OOF ROC-AUC = 0.96362


## 8. Save submission files

One file per model, plus one for the blend — the required format is `id,addicted_label`.


saved submission_xgboost_cv.csv  (shape=(296302, 2))
saved submission_lightgbm_cv.csv  (shape=(296302, 2))
saved submission_catboost_cv.csv  (shape=(296302, 2))
saved submission_blend.csv  (shape=(296302, 2))

In [25]:
def save_submission(name, test_ids, preds):
    sub = pd.DataFrame({"id": test_ids, "addicted_label": preds})
    filename = f"submission_{name}.csv"
    sub.to_csv(filename, index=False)
    print(f"saved {filename}  (shape={sub.shape})")

save_submission("xgboost_cv", test_df["id"], xgb_test_preds)
save_submission("lightgbm_cv", test_df["id"], lgbm_test_preds)
save_submission("catboost_cv", test_df["id"], cat_test_preds)
save_submission("blend", test_df["id"], blend_test_preds)


saved submission_xgboost_cv.csv  (shape=(296302, 2))
saved submission_lightgbm_cv.csv  (shape=(296302, 2))
saved submission_catboost_cv.csv  (shape=(296302, 2))
saved submission_blend.csv  (shape=(296302, 2))


## 9. Sanity check before you submit


In [26]:
check = pd.read_csv("submission_blend.csv")
print(check.columns.tolist())                          # [id, addicted_label]
print(check.shape)                                       # (296302, 2)
print(check["addicted_label"].between(0, 1).all())        # True
print(check["id"].isna().sum())                           # 0
check.head()


['id', 'addicted_label']
(296302, 2)
True
0


,id,addicted_label
0,691369,0.998679
1,691370,0.940241
2,691371,0.954079
3,691372,0.988249
4,691373,0.997994


## 10. If you're still not at the score you want

At this point the model side is essentially exhausted — check these, in order:

1. **Re-verify `ORDINAL_MAPS`** in Section 2 exactly matches the real category strings printed
   in the "unique values" cell — a wrong/missing key silently falls back to `NaN` -> median, which
   quietly throws away signal.
2. **Increase `N_TRIALS`** in Section 0 (e.g. 30 -> 100) for a finer Optuna search — diminishing
   returns, but every bit helps near the top.
3. **Try `N_FOLDS = 10`** for a more stable OOF estimate and averaged test prediction.
4. **Look for target leakage or near-duplicate rows** between train and test — if some real
   users appear in both with slightly different noise, that inflates leaderboard scores you're
   comparing yourself to but isn't reproducible cleanly.
5. **Add more engineered features** — this is the biggest remaining lever once 3 strong models
   already agree closely with each other (check `np.corrcoef(xgb_oof, cat_oof)` — if they're
   >0.98 correlated, they're already extracting the same signal, and the ceiling is in the
   *data*, not the *model*).
